<a href="https://colab.research.google.com/github/adnan855570/Langgraph/blob/main/Simple_LLM_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "langchain-openrouter"
!pip install langchain
!pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 18.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 50.4 MB/s  0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.50.1
    Uninstalling pydantic_core-2.50.1:
      Successfully uninstalled pydantic_core-2.50.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.14.1
    Uninstalling pydantic-2.14.1:
      Successfully uninstalled pydantic-2.14.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain-openrouter]


In [2]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from typing import TypedDict
from dotenv import load_dotenv

In [3]:
load_dotenv()

False

In [4]:
model = init_chat_model()

In [8]:
import os
# Read key from .env
with open('/content/.env', 'r') as f:
    for line in f:
        if 'OPENROUTER_API_KEY' in line or 'openrouter_key' in line:
            print('Found API key in .env file.')
            key_val = line.split('=')[1].strip().replace('"', '').replace("'", "")
            os.environ['OPENROUTER_API_KEY'] = key_val

# Initialize model using langchain-openrouter
# We will use 'openrouter/auto' or a popular free model like 'google/gemma-2-9b-it:free'
model = init_chat_model(
    model="google/gemma-2-9b-it:free",
    model_provider="openrouter"
)

Found API key in .env file.


In [9]:
# create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [10]:
def llm_qa(state: LLMState) -> LLMState:

  # extract the question from state
  question = state['question']

  # form a prompt
  prompt = f'Answer the following question {question}'

  # ask the question to the LLM
  answer = model.invoke(prompt).content

  # update the answer in the state
  state['answer'] = answer

  return state

In [11]:
# create a graph
graph = StateGraph(LLMState)

# add a node
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [12]:
# re-initialize model with the auto-routed free OpenRouter model
global model
model = init_chat_model("openrouter/free", model_provider="openrouter")

# execute
intial_state = {'question': 'How far is moon from the earth?'}

# workflow.invoke returns the final state dictionary
final_state = workflow.invoke(intial_state)
print("Answer:", final_state['answer'])

Answer: 

Of course! That's an excellent question with a more complex answer than you might expect, because the distance between the Earth and the Moon is always changing.

Here’s a breakdown of the answer:

### The Short Answer

On average, the Moon is about **384,400 kilometers (238,900 miles)** away from Earth.

### The More Detailed Answer

The Moon doesn't orbit the Earth in a perfect circle; it follows a slightly elliptical path. This means the distance between us and the Moon varies throughout its orbit, which takes about 27.3 days.

*   **Perigee (Closest Point):** The Moon is closest to Earth at a point called *perigee*. At this time, it can be as close as **363,300 kilometers (225,700 miles)**.
*   **Apogee (Furthest Point):** The Moon is farthest from Earth at a point called *apogee*. At this time, it can be as far away as **405,500 kilometers (251,900 miles)**.

The difference between perigee and apogee is about 42,200 km (26,200 miles).

### Why This Difference Matters

Th